> **구분: 수업 후 복습·자습**

# 03. Seq2Seq와 Attention — 30분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 현재 실습의 입력·저장·실행 범위

열 파일은 `/workspace/04.LLM/03_seq2seq_attention.ipynb`입니다. 첫 환경 셀은 `ROOT`, `COURSE`, `DATA`, `OUTPUTS`, `check`, `load_op`, `run_training`을 **이 커널에 다시 정의** 합니다. Python 경로와 PyTorch·CUDA 출력에서 CUDA가 `True`인지 확인하고 같은 Jetson의 다른 사용자와 GPU 실행 순서를 맞춥니다. 이 숫자 모델 훈련에는 Ollama 서버가 필요하지 않습니다.

입력은 `/workspace/Data/datasets/04.LLM/digits.json`과 `.sha256`입니다. 이 데이터 640개를 train 512개·validation 128개로 나누며, PAD/BOS/EOS=0/1/2, 숫자 0~9의 token ID=3~12입니다. 파일이 없으면 배포 데이터가 포함된 개인 자료 폴더인지 확인합니다. 이전 훈련 checkpoint 없이 새 훈련을 시작합니다.

`exercises/*.py`는 Jupyter 파일 편집기에서 수정·저장하세요. 노트북의 파일 출력 셀은 읽기 전용이고 `check(..., 'exercise')`는 저장한 최신 파일을 읽습니다. 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`·`loss.csv`, 자신의 가중치는 `/workspace/Data/checkpoints/users/04.LLM/`에 남습니다. `run_training`은 새 결과 폴더를 만들고 반환한 `report['checkpoint']`는 `Data` 기준 상대 경로입니다. 다른 노트북의 `report` 변수는 이어지지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

입력을 읽는 순서와 출력할 순서가 다를 때 encoder와 decoder의 역할을 나눌 수 있습니다. attention은 decoder가 매 시점에 입력의 어느 위치를 참고할지 계산하게 합니다. 역순 과제로 읽기와 생성의 연결을 눈으로 확인합니다.

### 전체 과정에서 내 코드가 하는 일

```text
encoder 입력 상태
  ↓
decoder 질의
  ↓
attention 가중치
  ↓
참고 정보 결합
  ↓
다음 token 예측
  ↓
자유 생성 평가
```

attention 축과 정답의 한 칸 이동을 구현합니다. teacher forcing은 학습 때 정답 이전 token을 주는 방식이며 실제 생성 때 정답이 주어지는 것은 아닙니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** teacher forcing 결과가 좋은데 자유 생성이 틀릴 수 있나요?

**설명:** 그렇습니다. 자유 생성은 자기 예측을 다음 입력으로 사용하므로 앞 오류가 뒤로 이어질 수 있습니다. attention 가중치 모양·정답 위치·평가 방식을 분리해 확인하고 학습 출력만으로 생성 성능을 결론내리지 않습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| 행렬 곱 shape 오류·attention 합 불일치 | Q/K/V 축, transpose와 softmax 축, decoder 입력 shift 및 BOS 위치를 검증합니다. | 해당 check(..., exercise) 통과 |
| `ModuleNotFoundError: common` / 다른 함수가 로딩됨 | 작업 경로와 커널을 확인합니다. 다른 과목 common이 남으면 저장 후 해당 커널을 재시작하고 초기 셀부터 실행합니다. | 현재 04.LLM 모듈 import 성공 |
| 훈련 timeout·OOM·loss NaN | 직전 stdout/stderr와 설정을 보존합니다. 작은 quick profile부터 확인하고 중복 실행·입력 유한성·device를 점검합니다. | 작은 훈련 완료, 유한 loss와 metrics 저장 |

**예시에 없는 오류를 확인할 곳:** [LLM README](README.md), [contracts 검사](common/contracts.py), exercises/solutions의 해당 연산. 실행이 출력한 output 경로의 metrics.json과 터미널 오류를 확인하세요. [PyTorch 이슈](https://github.com/pytorch/pytorch/issues)는 버전과 최소 tensor 예제로 검색합니다.


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

본수업 훈련 120분은 [00_training_core.ipynb](00_training_core.ipynb)에서 진행합니다. 이 노트북 전체는 강사 선택 시연·자습이며 여기에 적힌 시간은 별도 자습 예상 시간입니다. 다섯 모델 구현 완료는 본수업 통과 조건이 아닙니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 시간·목표

핵심 지도·teacher forcing 5분 → shift TODO 5분 → 최소 attention·TODO 8분 → 두 모델 훈련 8분 → 결과 확인 4분. 선행: 01 loss.

### 자습 진행 경로

1. 환경과 바로 아래 **30분 수업 핵심 지도** 를 읽습니다. 1–2절의 긴 설명은 수업 후 복습합니다.
2. 3절의 shift-right 표를 확인하고 shift TODO·검증을 완료합니다.
3. 4절에서 이 실습의 최소 attention 범위를 확인하고 attention TODO·검증을 완료합니다.
4. Seq2Seq와 Attention quick 훈련을 실행하고 5절의 자유 생성 결과를 비교합니다.
5. shape 실험과 6절 오류 표는 수업 후 복습합니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

### 앞선 loss TODO의 위치와 현재 필요한 내용

원래 설명은 [00_training_core.ipynb의 「수업 핵심 TODO — loss」](00_training_core.ipynb)와 [01_training_basics.ipynb의 「자습 구현 TODO — loss」](01_training_basics.ipynb)에 있습니다. 두 노트북이 편집하는 실제 파일은 **[exercises/loss.py](exercises/loss.py)** 하나입니다. 이전 커널을 열 필요 없이 이 파일의 `token_loss(logits, targets)`를 아래 계약으로 완성·저장하세요.

- 입력 logits는 raw 실수 점수 `[B,T,V]`, targets는 같은 장치의 정수 class 번호 `[B,T]`(값 `0..V-1`)입니다.
- 마지막 후보 축 V를 남겨 logits를 `[B*T,V]`로, targets를 같은 순서로 `[B*T]`로 펼칩니다. `reshape(-1, logits.shape[-1])`와 `reshape(-1)`의 역할을 구분합니다.
- `F.cross_entropy`에 두 값을 전달하고 평균 loss인 0차원 텐서를 반환합니다. softmax를 미리 적용하거나 `.item()`, `.detach()`로 gradient 연결을 끊지 않습니다.
- 모든 점수가 같은 후보 4개라면 loss는 약 `1.3863`이지만 이 값을 고정 반환하면 안 됩니다. 실제 검사에서는 정답 점수를 낮출 때 loss 증가와 backward의 비영 기울기까지 확인합니다.
- 이 노트북의 아래 `check('loss', 'exercise')`가 저장한 파일을 다시 읽습니다. 통과 후 훈련하세요. 참고 정답은 [solutions/loss.py](solutions/loss.py)에 있으며 이를 실행하는 것만으로 exercise 파일은 완성되지 않습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import importlib, json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.contracts import check
from common.loading import load_op
import torch
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '훈련은 Jetson GPU에서 수행합니다. 강사에게 환경 점검을 요청하세요.'
def run_training(kind, variant='exercise'):
    before = {p.resolve() for p in OUTPUTS.glob('*/metrics.json')}
    command = [sys.executable, str(COURSE / 'scripts/train.py'), '--model', kind, '--variant', variant]
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    created = [p for p in OUTPUTS.glob('*/metrics.json') if p.resolve() not in before]
    assert len(created) == 1, f'새 metrics.json이 정확히 1개여야 합니다: {created}'
    metrics_path = created[0]
    report = json.loads(metrics_path.read_text())
    assert report['kind'] == kind and report['variant'] == variant and report['profile'] == 'quick'
    report['_metrics_path'] = str(metrics_path.relative_to(DATA))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    if kind in ('rnn','lstm'):
        print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    else:
        print('자유 생성 token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('자유 생성 시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    print('실행 경로:', report['run_path'])
    return report


> **구분: 수업 후 복습·자습**

## 30분 수업 핵심 지도

- Encoder는 입력 전체를 읽고, decoder는 BOS부터 출력 token을 하나씩 만듭니다.
- Teacher forcing에서는 직전 정답을 decoder 입력으로 쓰되, 현재 답을 보지 않도록 target을 오른쪽으로 한 칸 옮깁니다.
- 기본 Seq2Seq는 encoder 마지막 state 하나에 입력을 압축합니다.
- Attention Seq2Seq는 매 decoder step마다 encoder의 모든 위치를 가중합해 context를 만듭니다.
- 이번 attention TODO는 projection이 없는 single-head 연산 `softmax(QKᵀ/√d_k)V`의 최소 구현입니다. 여기서는 `d_k=H`입니다.
- 훈련 loss는 teacher forcing 조건, token/sequence 정확도는 자기 예측을 다음 입력으로 넣는 자유 생성 조건입니다.


> **구분: 수업 후 복습·자습**

## 1. Sequence-to-Sequence 문제

[02_rnn_lstm.ipynb의 「이번 기억 과제」](02_rnn_lstm.ipynb)는 입력 위치마다 직전 입력을 출력하는 과제입니다. 예를 들어 숫자 `[1,2,3,4,5]`의 정답은 `[BOS,1,2,3,4]`이며, 이번 역순 과제와 정확도 순위를 비교하지 않습니다. 이번에는 입력 시퀀스 전체를 읽고 새로운 출력 시퀀스를 만드는 **Seq2Seq(sequence-to-sequence)** 구조를 다룹니다. 기계 번역, 요약, 음성 인식처럼 입력과 출력의 길이·순서가 다를 수 있는 문제의 기초입니다.

교육용 과제는 숫자 역순 변환입니다.

```text
입력 source : [3, 8, 5, 9, 4]       길이 5
정답 target : [4, 9, 5, 8, 3, EOS]  길이 6
```

EOS까지 정답에 포함시키는 이유는 모델이 내용뿐 아니라 **언제 출력을 끝낼지** 도 배워야 하기 때문입니다. 이 실습의 생성 코드는 비교를 단순화하려고 정답 길이만큼 반복하지만, 실제 생성 시스템은 EOS가 나오거나 최대 길이에 도달하면 멈춥니다.


> **구분: 수업 후 복습·자습**

## 2. Encoder와 Decoder

- **encoder** 는 source token을 순서대로 읽고 hidden representation을 만듭니다.
- **decoder** 는 BOS에서 시작해 target token을 한 개씩 생성합니다.

기본 Seq2Seq는 encoder의 마지막 state를 decoder의 초기 state로 전달합니다. 여러 페이지의 내용을 한 문장으로 기억해서 다시 쓰는 것처럼, 고정 길이 state 하나가 입력 전체를 요약해야 하므로 입력이 길면 병목이 생길 수 있습니다.

이 자료의 encoder와 decoder는 **GRU(Gated Recurrent Unit)** 를 사용합니다. GRU는 LSTM처럼 gate로 정보 흐름을 조절하지만 별도 cell state 없이 hidden state 하나를 사용합니다. LSTM보다 항상 낫거나 못한 구조라는 뜻은 아니며, 여기서는 encoder/decoder 흐름을 간단히 보기 위한 선택입니다.

기본 shape 흐름은 다음과 같습니다.

```text
source [B,5] → embedding [B,5,32] → encoder memory [B,5,64]
                                      └→ 마지막 state [1,B,64]
decoder input [B,6] → 매 시점 GRU → decoder state [B,1,64]
                                  → logits [B,1,V]를 6번 연결
                                  → 전체 logits [B,6,V]
```


> **구분: 수업 후 복습·자습**

## 3. Teacher forcing과 shift-right

<!-- procedure:lesson-13-01cbf5:start -->
![훈련과 생성의 입력이 다릅니다](../교재그림/절차/lesson-13.svg)

4번과 5번은 서로 다른 실행 경로입니다. teacher forcing loss만으로 자유 생성 성공을 판단하지 않습니다.
<!-- procedure:lesson-13-01cbf5:end -->


훈련 초기에 decoder의 예측은 거의 무작위입니다. 이전 예측만 계속 다음 입력으로 넣으면 첫 오류가 뒤의 모든 위치에 퍼져 학습 신호가 불안정해질 수 있습니다. **teacher forcing** 은 훈련할 때 직전의 정답 token을 다음 decoder 입력으로 주는 방법입니다.

현재 위치의 정답을 입력으로 주면 답을 미리 보는 **target leakage(정답 누수)** 가 됩니다. 따라서 target을 오른쪽으로 한 칸 이동합니다.

```text
위치             0  1  2  3  4  5
정답 target      4  9  5  8  3 EOS
decoder 입력   BOS  4  9  5  8  3
```

위치 0에서는 BOS만 보고 첫 정답 4를 예측합니다. 위치 1에서는 BOS와 직전 정답 4를 이용해 9를 예측합니다. EOS는 예측해야 할 정답이지만 그 뒤 위치가 없으므로 이번 decoder 입력에는 들어가지 않습니다.

Teacher forcing은 훈련 방법이지 모델 구조의 이름이 아닙니다. 추론 때는 정답을 알 수 없으므로 직전 **자기 예측** 을 넣습니다. 이 훈련과 추론 조건의 차이를 **exposure bias** 라고 부르며, 자유 생성 성능을 별도로 측정해야 하는 이유입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
target = torch.tensor([[4,9,5,8,3,2]])
print('target shape:', target.shape)
print('정답:', target.tolist())
print('필요한 decoder 입력: [[1,4,9,5,8,3]]')

> **구분: 수업 후 복습·자습**

## 자습 구현 TODO — shift

수정 파일: **[04.LLM/exercises/shift.py](exercises/shift.py)**

첫 위치는 BOS=1, 이후는 직전 정답으로 구성합니다. dtype·device·shape를 유지하고 입력을 수정하지 마세요.
힌트: `torch.empty_like`, `result[:, 0]`, `targets[:, :-1]`.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `shift_right(targets, bos=1)`은 T≥1인 정수 텐서 `[B,T]`를 받아 같은 shape·dtype·device의 새 텐서를 반환합니다. `[[4,5,6]]` → `[[1,4,5]]`; T=1이면 BOS 한 칸만 남습니다.
**작성 흐름 힌트:** 입력과 같은 빈 텐서 만들기 → 첫 열에 BOS → 나머지 열에 targets의 마지막 열을 제외한 값 복사. `targets` 자체를 수정하지 않습니다.
**설명할 지점:** “지금 맞힐 정답을 decoder에 그대로 보여 주지 않고, 직전 정답만 입력하는 teacher forcing 준비입니다.”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'exercises/shift.py').read_text())

In [ ]:
# 구분: 수업 후 복습·자습 셀
check('shift', 'exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습**

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/shift.py').read_text())

### 정답 해설 — 입력부터 결과까지

**정답을 한 칸 미루는 이유:** decoder가 현재 정답을 바로 입력으로 보면 답을 베낄 수 있습니다. 시작에는 BOS를 넣고 이후 위치에는 직전 정답을 입력합니다. targets `[4,5,6]`, BOS=1이면 decoder 입력은 `[1,4,5]`입니다.

`empty_like`로 같은 모양을 만들고 첫 열·나머지 열을 따로 채우므로 원본 targets는 바뀌지 않습니다. `[:,1:]`와 `[:,:-1]`의 위치가 대응합니다. teacher forcing은 훈련 시 이전 정답을 주는 방식이며 실제 자유 생성에서는 이전 모델 예측을 사용하므로 두 정확도는 다를 수 있습니다.

**확인 질문:** shift 없이 `[4,5,6]`을 그대로 주면 첫 출력 위치가 무엇을 미리 보게 되나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/shift.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import torch

def shift_right(targets, bos=1):
    # 원본 targets와 저장 공간을 공유하지 않는 같은 형상의 텐서를 만듭니다.
    result = torch.empty_like(targets)
    # 첫 입력은 시작 토큰이며 다음 위치부터 직전 정답을 받습니다.
    result[:, 0] = bos
    result[:, 1:] = targets[:, :-1]
    return result
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('shift', 'solution')
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습**

### shift TODO에서 확인할 네 가지

1. 반환 shape가 target과 같은 `[B,T]`인가?
2. 첫 열 전체가 BOS인가?
3. 두 번째 열부터가 원래 target의 마지막 열을 제외한 값인가?
4. 원본 target의 값, dtype과 device를 바꾸지 않았는가?

`result = targets`처럼 같은 객체를 받은 뒤 첫 열을 수정하면 원본 정답까지 바뀝니다. 정답 데이터가 조용히 손상되면 이후 loss가 이상해지므로 새 텐서를 만들어야 합니다. `torch.empty_like(targets)`는 같은 shape·dtype·device의 새 텐서를 만들되 값은 초기화하지 않으므로 모든 위치를 빠짐없이 채워야 합니다.


> **구분: 수업 후 복습·자습**

## 4. Attention은 입력의 어느 위치를 얼마나 섞을지 계산합니다

마지막 state 하나에 모든 입력을 담는 대신, decoder가 출력 한 개를 만들 때마다 encoder의 모든 위치를 다시 참고하게 할 수 있습니다. 도서관에서 질문에 맞는 책의 여러 페이지에 표시를 붙이고 필요한 내용을 모으는 것과 비슷하지만, 실제 계산은 학습된 벡터의 유사도와 가중합입니다.

- **Query(Q)** `[B,1,H]`: 현재 decoder가 찾고 있는 정보
- **Key(K)** `[B,T,H]`: 각 encoder 위치가 어떤 정보인지 비교할 표지
- **Value(V)** `[B,T,H]`: 선택 비중에 따라 실제로 가져올 내용

실제 Transformer attention은 보통 학습 가능한 선형 projection으로 입력에서 Q/K/V를 만들고 여러 head를 사용합니다. **이번 TODO는 원리를 분리해서 보기 위한 projection 없는 single-head 최소 구현** 입니다. Decoder GRU의 query를 그대로 Q로, encoder memory를 K와 V로 그대로 사용합니다.

계산은 다음 순서입니다.

1. `K.transpose(1,2)`는 `[B,T,H] → [B,H,T]`로 바뀝니다.
2. `Q @ Kᵀ`는 `[B,1,H] @ [B,H,T] → score [B,1,T]`입니다.
3. 일반식은 key/query 한 head의 차원 `d_k`를 사용해 `sqrt(d_k)`로 나눕니다. 이 최소 구현은 projection이나 head 분할이 없어 `d_k=H`이므로 코드에서는 `sqrt(query.shape[-1])`입니다.
4. 입력 위치인 마지막 축 `T`에 softmax를 적용해 `weights [B,1,T]`를 만듭니다.
5. `weights @ V`는 `[B,1,T] @ [B,T,H] → context [B,1,H]`입니다.

각 query에서 attention weights의 합은 1입니다. 이 값은 **최종 출력 token의 확률** 이 아니라 value 벡터를 섞는 비중입니다. `Seq2Seq.step`은 계산한 `context [B,1,H]`를 decoder의 원래 `query [B,1,H]`와 마지막 축으로 이어 붙여 `[B,1,2H]`로 만든 뒤, 출력 Linear 층으로 logits `[B,1,V]`를 계산합니다. 즉, context가 query를 대체하는 방식이 아니라 **query-context concat 방식** 입니다.

높은 attention weight를 곧바로 사람이 이해하는 원인이나 설명으로 단정해서는 안 됩니다. 같은 token이 여러 번 나오거나 정보가 여러 위치에 분산되면 한 위치만 선택할 필요도 없습니다.


> **구분: 수업 후 복습·자습**

## 자습 구현 TODO — attention

수정 파일: **[04.LLM/exercises/attention.py](exercises/attention.py)**

query `[B,1,H]`, keys/values `[B,T,H]`에서 context `[B,1,H]`와 weights `[B,1,T]`를 반환합니다.
힌트: `torch.bmm`, `transpose(1,2)`, `softmax(dim=-1)`를 연결하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `scaled_attention(query, keys, values)`는 query `[B,1,H]`, keys와 values `[B,T,H]`를 받아 `(context, weights)` 튜플을 반환합니다. 각각 `[B,1,H]`, `[B,1,T]`입니다.
**작성 흐름 힌트:** query와 전치한 keys의 batched 곱 → **sqrt(H)로 나누어 점수 스케일 조정** → 시간축 T에 softmax → weights와 values의 batched 곱. 이 최소 구현은 projection/head 분할이 없어 d_k=H입니다.
**손으로 확인:** query가 0이고 T=2면 weights는 `[0.5,0.5]`, context는 두 value의 평균입니다. 각 배치의 weights 합은 1입니다.
**설명할 지점:** “어느 입력 위치를 참고할지 정하는 weights와, 실제로 섞어 가져오는 values의 역할을 나눠 봅시다.”
<!-- todo-review:requirements:end -->

**추가 경계와 검증:** 세 입력은 같은 dtype·device이고 B, T, H는 양수인 실수 텐서입니다. 입력을 바꾸거나 `.detach()`로 기울기를 끊지 않습니다. `(context, weights)` 순서로 반환하며 마지막 softmax 축은 H가 아니라 입력 위치 T입니다. 아래 검증에서 query의 gradient, scale `sqrt(H)`, 가중합 값, 동일 점수의 균등 가중치까지 확인합니다. `torch.bmm(query, keys.transpose(1,2))`가 만드는 shape는 `[B,1,T]`입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'exercises/attention.py').read_text())

In [ ]:
# 구분: 수업 후 복습·자습 셀
check('attention', 'exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습**

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/attention.py').read_text())

### 정답 해설 — 입력부터 결과까지

**점수→가중치→합:** query와 각 key의 내적으로 관련 점수를 계산하고 sqrt(H)로 크기를 조절합니다. 이 최소 구현에는 head 분할이 없어 H가 key 차원입니다. 마지막 입력 위치 축으로 softmax를 취하면 가중치 합이 1이 됩니다. 그 가중치로 values를 섞어 context를 만듭니다.

두 점수가 같고 values가 10과 20이라면 가중치는 각각 0.5, context는 15입니다. `[B,Tq,H]`와 `[B,Tk,H]`에서 keys를 transpose해 `[B,H,Tk]`로 만든 뒤 bmm하면 `[B,Tq,Tk]`가 됩니다.

**확인 질문:** softmax를 잘못된 축에 적용해도 tensor 계산이 될 수 있습니다. 어느 축의 합이 1인지 검사해야 하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/attention.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import math
import torch

def scaled_attention(query, keys, values):
    # 일반식의 d_k. 이 최소 구현은 projection/head 분할이 없어 d_k == H입니다.
    scores = torch.bmm(query, keys.transpose(1,2)) / math.sqrt(query.shape[-1])
    # 마지막 축 T에 정규화하여 배치마다 참고 위치의 가중치 합을 1로 만듭니다.
    weights = scores.softmax(dim=-1)
    # [B,1,T] @ [B,T,H] → [B,1,H]; 가중치도 반환해 해석할 수 있게 합니다.
    return torch.bmm(weights, values), weights
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('attention', 'solution')
print('solution_verification=PASS')

> **구분: 수업 후 복습·자습**

### attention shape를 손으로 예측한 뒤 확인하기

아래 예시는 계산 구조만 확인하기 위해 참고 구현을 사용합니다. `B=2`, 입력 길이 `T=5`, hidden 크기 `H=4`입니다. 실행 전에 context와 weights의 shape를 먼저 종이에 적으세요.


In [ ]:
# 구분: 수업 후 복습·자습 셀
attention_solution = load_op('attention', 'solution')
q = torch.randn(2, 1, 4)
k = torch.randn(2, 5, 4)
v = torch.randn(2, 5, 4)
context_demo, weights_demo = attention_solution.scaled_attention(q, k, v)
print('context:', context_demo.shape)
print('weights:', weights_demo.shape)
print('샘플별 weight 합:', weights_demo.sum(dim=-1))


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('loss', 'exercise')
check('shift', 'exercise')
check('attention', 'exercise')
seq_report = run_training('seq2seq')
attention_report = run_training('attention')

> **구분: 수업 후 복습·자습**

## 5. teacher forcing loss와 실제 생성은 다릅니다

검증 loss는 정답 문맥을 준 상태에서 측정합니다. 보고서의 token/sequence 정확도는 **자신의 예측을 다음 입력으로 사용하는 자유 생성** 으로 측정합니다.
loss가 낮아도 자유 생성에서는 앞부분의 오류가 다음 입력으로 들어가 뒤의 오류를 만들 수 있습니다. 따라서 teacher-forced loss와 자유 생성 정확도가 다르게 움직일 수 있습니다.

결과의 `target`과 `predicted`를 왼쪽부터 비교하세요. 앞 위치에서 처음 틀린 뒤 나머지가 연속으로 틀렸다면 오류 전파 가능성이 있습니다. token accuracy가 높지만 sequence accuracy가 낮다면 대부분 맞아도 한두 위치가 자주 틀린다는 뜻일 수 있습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
for report in (seq_report, attention_report):
    print('\n모델:', report['kind'])
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    print('자유 생성 token 정확도:', report['after']['token_accuracy'])
    print('자유 생성 sequence 정확도:', report['after']['sequence_accuracy'])
    for example in report['after']['examples']:
        print('입력', example['input'], '정답', example['target'], '예측', example['predicted'])

> **구분: 수업 후 복습·자습**

## 6. 자주 생기는 오해와 오류

- **"teacher forcing이면 모델이 cheating한 것이다"** : 올바르게 한 칸 이동한 이전 정답만 제공하는 것은 표준적인 훈련 방식입니다. 현재 정답을 같은 위치 입력으로 주는 것이 누수입니다.
- **"attention weight가 가장 큰 위치가 모델 판단의 완전한 이유다"** : weight는 해당 층의 value 혼합 계수입니다. 다른 층, residual 경로와 출력 변환도 결과에 관여합니다.
- **attention softmax 축 오류** : `H`가 아니라 입력 위치 `T` 축인 마지막 축에 적용해야 각 query가 위치별 비중을 만들 수 있습니다.
- **`bmm` shape 오류** : `torch.bmm`는 3차원 batch 행렬곱입니다. `Q [B,1,H]`와 전치한 `K [B,H,T]`인지 확인합니다.
- **decoder 입력 길이 오류** : target과 shift-right 결과는 모두 `[B,6]`이어야 합니다.
- **원본 target이 바뀜** : in-place 수정 여부를 확인하고 TODO 검증을 다시 실행합니다.
- **Seq2Seq 결과가 Attention보다 낮음** : 이 seed와 짧은 설정의 관찰입니다. 모든 데이터와 설정에서의 우열로 일반화하지 않습니다.


> **구분: 수업 후 복습·자습**

## 완료·자습

shift·attention 검증과 두 모델 훈련을 완료하고, 고정 context와 attention의 차이를 설명합니다.
자습(20분): 입력 token이 모두 같을 때의 attention과 생성 결과를 관찰하세요. quick과 selfstudy 설정의 정확도·시간을 비교합니다.
단일 seed 결과만으로 특정 구조가 항상 우수하다고 결론내리지 않습니다.
